<a href="https://colab.research.google.com/github/NiloyKumarKundu/Graph-Neural-Network/blob/main/3_Graph_Attention_Network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Single Head

In [5]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

# ---------- 1. Toy graph ----------
H = {
    "A": np.array([1.0, 0.0]),
    "B": np.array([2.0, 1.0]),
    "C": np.array([0.0, 1.0]),
}
# A-এর neighbour, self-loop সহ (A নিজেকেও শোনে)
neighbors = {"A": ["A", "B", "C"]}

# ---------- 2. Learnable parameter (training শেখে; এখানে হাতে বসানো) ----------
W = np.array([[1.0, 0.0],
              [1.0, 1.0]])            # shape (F'=2, F=2)
a = np.array([0.0, 0.0, 2.0, -1.0])   # attention vector, length 2F'=4

# ---------- helper ----------
def leaky_relu(x, slope=0.2):
    return np.maximum(x, slope * x)

def softmax(scores):
    ex = np.exp(scores)
    return ex / ex.sum()

# ---------- 3. GAT layer, এক target node-এর জন্য ----------
def gat_layer(target, neighbors, H, W, a):
    # STEP 1 — shared linear transform: z = W h
    nodes = set([target] + neighbors[target])
    print(f"Nodes: {nodes}")
    z = {n: W @ H[n] for n in nodes}
    print(f"z: {z}")

    # STEP 2 — প্রতিটা neighbour-এর raw score
    nbrs = neighbors[target]
    print(f"nbrs: {nbrs}")
    scores = []
    for j in nbrs:
        concat = np.concatenate([z[target], z[j]])   # [z_i || z_j]
        raw = a @ concat                             # scalar dot product
        e = leaky_relu(raw)
        print(f"e: {e}")
        scores.append(e)
    scores = np.array(scores)
    print(f"scores: {scores}")

    # STEP 3 — softmax → attention weight alpha
    alpha = softmax(scores)
    print(f"alpha: {alpha}")

    # STEP 4 — weighted sum + ReLU
    out = np.zeros_like(z[target])
    print(f"out: {out}")
    for j, al in zip(nbrs, alpha):
        out += al * z[j]
    print(f"out: {out}")
    return np.maximum(0, out)

gat_layer("A", neighbors, H, W, a)

Nodes: {'B', 'A', 'C'}
z: {'B': array([2., 3.]), 'A': array([1., 1.]), 'C': array([0., 1.])}
nbrs: ['A', 'B', 'C']
e: 1.0
e: 1.0
e: -0.2
scores: [ 1.   1.  -0.2]
alpha: [0.4346 0.4346 0.1309]
out: [0. 0.]
out: [1.3037 1.8691]


array([1.3037, 1.8691])

## Multi-Head

In [7]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

# ---------- 1. Toy graph (self-loop সহ) ----------
H = {
    "A": np.array([1.0, 0.0]),
    "B": np.array([2.0, 1.0]),
    "C": np.array([0.0, 1.0]),
}
neighbors = {"A": ["A", "B", "C"]}

# ---------- 2. Per-head parameter (প্রতি head-এর নিজের W আর a) ----------
heads = [
    {"W": np.array([[1.0, 0.0],
                    [1.0, 1.0]]),
     "a": np.array([0.0, 0.0, 2.0, -1.0])},      # head 1 (চেনা head)
    {"W": np.array([[0.0, 1.0],
                    [1.0, 0.0]]),
     "a": np.array([0.0, 0.0, 1.0, 1.0])},       # head 2 (আলাদা)
]

def leaky_relu(x, slope=0.2): return np.maximum(x, slope * x)
def softmax(s):
    ex = np.exp(s); return ex / ex.sum()

# ---------- এক head: pre-activation aggregate (Σ α·z) ফেরায় ----------
def one_head(target, nbrs, H, W, a, head_id):
    z = {n: W @ H[n] for n in set([target] + nbrs)}        # STEP 1: project
    scores = np.array([leaky_relu(a @ np.concatenate([z[target], z[j]]))
                       for j in nbrs])                     # STEP 2: raw scores
    alpha = softmax(scores)                                # STEP 3: softmax
    agg = sum(al * z[j] for j, al in zip(nbrs, alpha))     # STEP 4: weighted sum
    return agg

# ---------- multi-head GAT layer ----------
def multihead_gat(target, neighbors, H, heads, mode="concat"):
    nbrs = neighbors[target]
    aggregates = [one_head(target, nbrs, H, hp["W"], hp["a"], k)
                  for k, hp in enumerate(heads, start=1)]

    if mode == "concat":                       # hidden layer style
        out = np.concatenate([np.maximum(0, agg) for agg in aggregates])
    else:                                      # output layer style
        out = np.maximum(0, np.mean(aggregates, axis=0))
    return out

In [8]:
multihead_gat("A", neighbors, H, heads)

array([1.3037, 1.8691, 0.8935, 1.6805])